# FARZ — Export the trained model to ExecuTorch for Android

This notebook does **not retrain**. It takes the trained MobileNetV3-Small checkpoint and the exact Experiment 01 `data_split.csv`, downloads the original resized images directly into Colab, verifies every image SHA-256, exports the `.pte` using ExecuTorch 1.5.1 + XNNPACK, tests parity, and packages the model for the FARZ Android app.

Upload these three files when prompted: `FARZ-ML-Training-Source.zip`, `FARZ-training-results.zip`, and `data_split.csv`.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, shutil, json, subprocess, sys, csv, hashlib, os

uploaded = files.upload()
source_zip = next((n for n in uploaded if n.lower().endswith('.zip') and 'ml-training-source' in n.lower()), None)
results_zip = next((n for n in uploaded if n.lower().endswith('.zip') and 'training-results' in n.lower()), None)
split_csv = next((n for n in uploaded if n.lower() == 'data_split.csv'), None)
if not all([source_zip, results_zip, split_csv]):
    raise FileNotFoundError('Upload FARZ-ML-Training-Source-v6.zip, FARZ-training-results.zip, and data_split.csv')
PROJECT = Path('/content/FARZ-AI')
if PROJECT.exists(): shutil.rmtree(PROJECT)
with zipfile.ZipFile(source_zip) as z: z.extractall('/content')
if not (PROJECT / 'ml/src/export/export_executorch.py').is_file():
    raise FileNotFoundError('Source bundle lacks the ExecuTorch exporter; download FARZ-ML-Training-Source-v6.zip.')
RESULTS = Path('/content/farz-results')
if RESULTS.exists(): shutil.rmtree(RESULTS)
RESULTS.mkdir()
with zipfile.ZipFile(results_zip) as z: z.extractall(RESULTS)
checkpoint = next(RESULTS.rglob('best_model.pth'), None)
if checkpoint is None: raise FileNotFoundError('best_model.pth not found in FARZ-training-results.zip')
manifest = PROJECT / 'data/manifests/data_split.csv'
manifest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(split_csv, manifest)
checkpoint_target = PROJECT / 'ml/outputs/mobilenet_v3_small/best_model.pth'
checkpoint_target.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(checkpoint, checkpoint_target)
os.chdir(PROJECT)
print('Source ready:', PROJECT, '| Checkpoint:', checkpoint_target.stat().st_size, 'bytes')


## 1. Install the matched ExecuTorch exporter

FARZ pins the Python exporter and Android Maven runtime to `1.5.1`. The ExecuTorch Python package does not install PyTorch automatically, so this notebook keeps Colab's existing PyTorch build and prints its version.

In [ ]:
%pip -q install --upgrade torch==2.14.0+cpu torchvision==0.29.0+cpu executorch==1.5.1 --index-url https://download.pytorch.org/whl/cpu --extra-index-url https://pypi.org/simple

import torch, torchvision, executorch
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('ExecuTorch:', getattr(executorch, '__version__', '1.5.1'))
assert torch.__version__.split('+')[0] == '2.14.0', f'Expected torch 2.14.0, got {torch.__version__}'
assert torchvision.__version__.split('+')[0] == '0.29.0', f'Expected torchvision 0.29.0, got {torchvision.__version__}'


## 2. Download the images directly into Colab and verify the exact split

The notebook does not regenerate a split. It checks the uploaded experiment manifest against the source image SHA-256 values.

In [ ]:
DATA_URL = 'https://huggingface.co/datasets/garythung/trashnet/resolve/main/dataset-resized.zip'
zip_path = Path('/content/dataset-resized.zip'); raw = PROJECT / 'data/raw'; raw.mkdir(parents=True, exist_ok=True)
if not zip_path.exists(): subprocess.run(['wget','-q','--show-progress','-O',str(zip_path),DATA_URL],check=True)
subprocess.run(['unzip','-q','-o',str(zip_path),'-d',str(raw)],check=True)
images = raw / 'dataset-resized'
rows = list(csv.DictReader(manifest.open(newline='',encoding='utf-8-sig')))
missing, mismatched = [], []
for row in rows:
    path = images / row['relative_path']
    if not path.is_file(): missing.append(row['relative_path']); continue
    if hashlib.sha256(path.read_bytes()).hexdigest() != row['sha256']: mismatched.append(row['relative_path'])
counts = {s:sum(r['split']==s for r in rows) for s in ['train','validation','test']}
print('Images:',len(rows),'| split:',counts,'| missing:',len(missing),'| hash mismatches:',len(mismatched))
assert len(rows)==2384 and counts=={'train':1705,'validation':339,'test':340}
assert not missing and not mismatched

## 3. Export `.pte` using XNNPACK and verify prediction parity

The exporter lowers the graph to XNNPACK, runs the resulting PTE with the ExecuTorch Python runtime, and checks that PyTorch and ExecuTorch choose the same top-1 class on 30 test images sampled across the five classes. The export is rejected if any selected top-1 result differs.

In [ ]:
subprocess.run([sys.executable,'ml/src/export/export_executorch.py',
    '--root','.', '--checkpoint','ml/outputs/mobilenet_v3_small/best_model.pth',
    '--manifest','data/manifests/data_split.csv', '--images-dir','data/raw/dataset-resized',
    '--output','ml/outputs/mobilenet_v3_small/farz_model.pte', '--parity-images','30', '--copy-to-app'],
    cwd=PROJECT,check=True)
export_dir=PROJECT/'ml/outputs/mobilenet_v3_small'
report=json.loads((export_dir/'executorch_export_report.json').read_text())
print(json.dumps(report,indent=2))
assert report['top1_match_rate']==1.0

## 4. Save the export package to Google Drive and download it

The package includes the `.pte`, preprocessing metadata, export report, parity predictions, exact split manifest, and source checkpoint. The `.pte` itself is the only required model asset for Android inference.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
package=Path('/content/FARZ-ExecuTorch-Export')
if package.exists(): shutil.rmtree(package)
package.mkdir()
for name, source in [
 ('farz_model.pte', PROJECT/'app/assets/models/farz_model.pte'),
 ('farz_model_metadata.json', export_dir/'farz_model_metadata.json'),
 ('executorch_export_report.json', export_dir/'executorch_export_report.json'),
 ('executorch_parity_predictions.json', export_dir/'executorch_parity_predictions.json'),
 ('data_split_run_01.csv', manifest),
 ('best_model.pth', checkpoint_target)]:
    if not source.is_file(): raise FileNotFoundError(source)
    shutil.copy2(source,package/name)
zip_out=shutil.make_archive('/content/FARZ-ExecuTorch-Export','zip',root_dir=package)
drive_out=Path('/content/drive/MyDrive/FARZ-Results/ExecuTorch-Android-1.5.1')
drive_out.mkdir(parents=True,exist_ok=True)
for item in package.iterdir(): shutil.copy2(item,drive_out/item.name)
print('Export package:',zip_out,'| size:',Path(zip_out).stat().st_size,'bytes')
files.download(zip_out)